# 01 — Prepare official Kroger API locations

Download or load Kroger locations from Kroger's official Locations API, validate them, and write the portable state, ZIP, and brand summaries. This notebook does not silently fall back to OpenStreetMap data.

In [1]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "scripts" / "build_all.py").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT / "scripts"))
ROOT

WindowsPath('D:/Healthy_food/Kroger_National_Coverage/kroger_national_coverage')

## Official API download

Create `.env` from `.env.example` and add your Kroger developer credentials. Set `RUN_OFFICIAL_DOWNLOAD = True` to refresh the data. The downloader uses OAuth2, respects Kroger's daily request limit, and never writes credentials to output files.

In [2]:
import subprocess, sys

RUN_OFFICIAL_DOWNLOAD = False  # Keep False when notebook 00 already succeeded
print(f'[1/5] API refresh requested: {RUN_OFFICIAL_DOWNLOAD}')

if RUN_OFFICIAL_DOWNLOAD:
    print('[2/5] Starting official Kroger API download...')
    result = subprocess.run(
        [sys.executable, str(ROOT / 'scripts' / 'download_official_api.py'), '--chain', 'KROGER'],
        cwd=ROOT,
    )
    if result.returncode:
        raise RuntimeError('Official Kroger API download failed. Check your .env credentials.')
    print('[2/5] Official API download complete.')
else:
    print('[2/5] Download skipped; using the successful output from notebook 00.')

[1/5] API refresh requested: False
[2/5] Download skipped; using the successful output from notebook 00.


In [3]:
import pandas as pd
from build_all import write_summaries

print('[3/5] Checking for the official API output...')
official_path = ROOT / 'data' / 'processed' / 'kroger_official_locations.csv'
if not official_path.exists():
    raise FileNotFoundError(
        'Official API export not found. Configure .env, set RUN_OFFICIAL_DOWNLOAD=True, and run the previous cell.'
    )
print('[3/5] Official API output found. Loading and validating records...')
locations = pd.read_csv(official_path, dtype={'zip_code': 'string'}).fillna({'website':'', 'phone':'', 'address':'', 'city':''})
assert locations['source'].eq('Kroger official Locations API').all(), 'Non-official records detected'
print(f'[4/5] Validation passed: {len(locations):,} official locations across {locations.state.nunique()} states.')
state_summary, zip_summary, brand_summary = write_summaries(locations)
print(f'[5/5] Complete: wrote {len(state_summary)} state rows, {len(zip_summary):,} ZIP rows, and {len(brand_summary)} brand rows.')
locations.head()

[3/5] Checking for the official API output...
[3/5] Official API output found. Loading and validating records...
[4/5] Validation passed: 1,264 official locations across 16 states.
[5/5] Complete: wrote 16 state rows, 1,023 ZIP rows, and 1 brand rows.


,location_id,name,brand,address,city,state,zip_code,phone,latitude,longitude,website,is_kroger_banner,status,status_basis,source,api_collected_at_utc
0,01100260,Kroger - Corner Village,KROGER,300 N Dean Rd,Auburn,AL,36830,3348211325.0,32.610565,-85.463195,,True,active,Returned by Kroger official Locations API on 2...,Kroger official Locations API,2026-09-01T18:57:47.243730+00:00
1,02600854,Kroger - Decatur,KROGER,1101 Beltline RD SE STE A,Decatur,AL,35601,2563509770.0,34.560158,-86.976354,,True,active,Returned by Kroger official Locations API on 2...,Kroger official Locations API,2026-09-01T18:57:47.243730+00:00
2,02600894,Kroger - Crestwood Shopping Center,KROGER,241 Highway 31 SW Ste K,Hartselle,AL,35640,2567736413.0,34.439458,-86.942830,,True,active,Returned by Kroger official Locations API on 2...,Kroger official Locations API,2026-09-01T18:57:47.243730+00:00
3,02600517,Kroger - Kroger Moores Mill Road Huntsville,KROGER,6070 Moores Mill Rd,Huntsville,AL,35811,2568525770.0,34.800504,-86.534893,,True,active,Returned by Kroger official Locations API on 2...,Kroger official Locations API,2026-09-01T18:57:47.243730+00:00
4,02600508,Kroger - Kroger Oakwood Ave Huntsville,KROGER,2110 Oakwood Ave Nw,Huntsville,AL,35810,2565360715.0,34.750633,-86.600692,,True,active,Returned by Kroger official Locations API on 2...,Kroger official Locations API,2026-09-01T18:57:47.243730+00:00


## Quality checks

Missing values are reported explicitly. A missing ZIP is retained in the location file but excluded from the ZIP summary.

In [4]:
locations[['location_id','brand','state','zip_code','status','status_basis','latitude','longitude']].isna().sum().to_frame('missing')

,missing
location_id,0
brand,0
state,0
zip_code,0
status,0
status_basis,0
latitude,0
longitude,0


## Store status

`active` means the source currently maps the record as `shop=supermarket`. It is not a real-time confirmation from Kroger. Lifecycle-tagged closed/disused objects are not present in this source export.

In [5]:
locations.groupby(['status', 'status_basis'], dropna=False).size().to_frame('locations')

,,locations
status,status_basis,
active,Returned by Kroger official Locations API on 2026-09-01,1264


In [6]:
locations.loc[locations['zip_code'].eq(''), ['name','brand','city','state','website']].head(20)

,name,brand,city,state,website


In [7]:
assert locations['state'].str.fullmatch(r'[A-Z]{2}').all()
assert locations['latitude'].between(18, 72).all()
assert locations['longitude'].between(-180, -60).all()
print('Basic U.S. coordinate and state checks passed.')

Basic U.S. coordinate and state checks passed.
